# Learning process diagnostics from saved checkpoints

This notebook is for the poster figure showing **what the diffusion model learns as training progresses**.

The previous image-grid view was useful for debugging, but the poster needs a more direct scientific diagnostic. Here we keep the sampler fixed and compare generated-field statistics from early, middle, and final training checkpoints.

Important setup:

- Same run: `nf_fig2_u128_d2p15_noaug_200k`
- Same inference method for every checkpoint: **DPM-Solver multistep, 50 denoising steps** (`dpm50`)
- Same generated-sample seed set, when available
- Only the saved model checkpoint changes, so the x-axis/labels mean **training time**, not a different sampler

Expected Great Lakes sampling command:

```bash
RUN_NAME=nf_fig2_u128_d2p15_noaug_200k \
EPOCHS=auto \
NUM_SAMPLES=6 \
MAX_PLOT_SAMPLES=5 \
SAMPLE_LABEL=dpm50 \
SAMPLER_CLASS=DPMSolverMultistepScheduler \
SAMPLER_STEPS=50 \
OVERWRITE=1 \
sbatch -A huterer2 scripts/slurm/sample_nf_generalize_epoch_snapshots.sbatch
```


In [ ]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
    Path('/Users/apple/diffusion-models-simulation-data'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram

RUN_NAME = os.environ.get('EPOCH_SNAPSHOT_RUN', 'nf_fig2_u128_d2p15_noaug_200k')
SAMPLE_LABEL = os.environ.get('SAMPLE_LABEL', 'dpm50')
SAMPLER_NOTE = os.environ.get('SAMPLER_NOTE', 'DPM-Solver, 50 steps')
SWEEP_NAME = os.environ.get('NF_GENERALIZE_SWEEP_NAME', 'nf_generalize_fig2')
SNAPSHOT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'epoch_snapshots'
OUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'quickcheck'
POSTER_DIR = PROJECT_DIR / 'poster' / 'figs'
CONFIG_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'configs' / f'{RUN_NAME}.yaml'

MAX_REAL_RAW_CUBES = int(os.environ.get('NF_EPOCH_MAX_REAL_RAW_CUBES', 16))
MAX_REAL_HIST = int(os.environ.get('NF_EPOCH_MAX_REAL_HIST', 2048))
MAX_REAL_PK = int(os.environ.get('NF_EPOCH_MAX_REAL_PK', 512))
PK_NBINS = int(os.environ.get('NF_EPOCH_PK_NBINS', 36))
HIST_BINS = int(os.environ.get('NF_EPOCH_HIST_BINS', 140))

OUT_DIR.mkdir(parents=True, exist_ok=True)
POSTER_DIR.mkdir(parents=True, exist_ok=True)

print('project:', PROJECT_DIR)
print('run:', RUN_NAME)
print('sampler:', SAMPLER_NOTE, f'({SAMPLE_LABEL})')
print('snapshots:', SNAPSHOT_DIR)
print('config:', CONFIG_PATH, 'exists=', CONFIG_PATH.exists())
print('poster output:', POSTER_DIR)


## Load checkpoint snapshots and the real reference

Each `.npz` file below is a generated sample set from one saved checkpoint. The samples should already have been generated with the same `dpm50` sampler. The real reference is loaded from the same run config and is used only for statistics, not for training here.


In [ ]:
def load_sample_array(path: Path) -> np.ndarray:
    data = np.load(path)
    if isinstance(data, np.lib.npyio.NpzFile):
        for key in ('samples', 'images', 'arr_0'):
            if key in data:
                arr = data[key]
                break
        else:
            arr = data[data.files[0]]
        data.close()
    else:
        arr = data
    arr = np.asarray(arr)
    if arr.ndim == 4:
        if arr.shape[1] in (1, 3):
            arr = arr[:, 0]
        elif arr.shape[-1] in (1, 3):
            arr = arr[..., 0]
        else:
            raise ValueError(f'Cannot infer image channel axis for {path}: shape={arr.shape}')
    if arr.ndim != 3:
        raise ValueError(f'Expected samples shaped (N,H,W), got {arr.shape} from {path}')
    return arr.astype(np.float32)


def to_nchw(images: np.ndarray) -> np.ndarray:
    arr = np.asarray(images)
    if arr.ndim == 3:
        return arr[:, None, :, :]
    if arr.ndim == 4 and arr.shape[1] == 1:
        return arr
    if arr.ndim == 4 and arr.shape[-1] == 1:
        return np.moveaxis(arr, -1, 1)
    raise ValueError(f'Expected (N,H,W) or (N,1,H,W), got {arr.shape}')


def epoch_from_name(path: Path) -> int:
    m = re.search(r'_epoch(\d+)_', path.name)
    if not m:
        raise ValueError(f'Could not parse epoch from {path.name}')
    return int(m.group(1))


def discover_snapshot_files() -> list[Path]:
    pattern = f'{RUN_NAME}_epoch*_seed*_{SAMPLE_LABEL}.npz'
    files = sorted(SNAPSHOT_DIR.glob(pattern), key=epoch_from_name)
    if not files:
        print('No snapshot files found. Run this on Great Lakes:')
        print(f'  RUN_NAME={RUN_NAME} EPOCHS=auto NUM_SAMPLES=6 SAMPLE_LABEL={SAMPLE_LABEL} OVERWRITE=1 sbatch -A huterer2 scripts/slurm/sample_nf_generalize_epoch_snapshots.sbatch')
        return []
    return files


def evenly_limit(arr: np.ndarray, n: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if n is None or len(arr) <= n:
        return arr
    idx = np.linspace(0, len(arr) - 1, int(n)).round().astype(int)
    return arr[idx]

files = discover_snapshot_files()
snapshots = []
for path in files:
    arr = load_sample_array(path)
    snapshots.append({'epoch': epoch_from_name(path), 'samples': arr, 'path': path})

if snapshots:
    print('found snapshots:')
    for row in snapshots:
        print(f"  epoch {row['epoch']:04d}: {row['samples'].shape}, {row['path'].name}")
else:
    print('No snapshot diagnostics can be plotted until files exist.')

if CONFIG_PATH.exists():
    real = as_nchw(load_real_from_config(CONFIG_PATH, max_raw_samples=MAX_REAL_RAW_CUBES))
    print('real reference:', real.shape)
else:
    real = None
    print('Missing config; image-only plots can still run, but statistics need real reference.')


## Stage labels

Raw epoch numbers are useful for provenance, but for the poster the simpler labels **early**, **middle**, **late**, and **final** are easier to read. The table below records the mapping.

In [ ]:
def stage_labels(n: int) -> list[str]:
    if n <= 0:
        return []
    if n == 1:
        return ['final']
    if n == 2:
        return ['early', 'final']
    if n == 3:
        return ['early', 'middle', 'final']
    if n == 4:
        return ['early', 'middle', 'late', 'final']
    labels = ['early']
    labels += [f'mid {i}' for i in range(1, n - 1)]
    labels += ['final']
    return labels

for label, row in zip(stage_labels(len(snapshots)), snapshots):
    row['stage'] = label

stage_table = pd.DataFrame([
    {'stage': row['stage'], 'epoch': row['epoch'], 'n_generated': len(row['samples']), 'file': row['path'].name}
    for row in snapshots
])
display(stage_table)


## Metrics: one-point PDF and spatial power spectrum

The main question is whether the model learns broad spatial structure before small-scale detail. The P(k) ratio panel makes that visible:

- low k: large spatial scales
- high k: small spatial scales
- ratio near 1: generated fields match the real reference

All curves use the same DPM-Solver-50 sampler; only checkpoint weights differ.

In [ ]:
def histogram_density(images: np.ndarray, bin_edges: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    vals = np.asarray(images, dtype=np.float64).ravel()
    hist, edges = np.histogram(vals, bins=bin_edges, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist


def smooth(y: np.ndarray, width: int = 5) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if width <= 1 or len(y) < width:
        return y
    kernel = np.ones(width, dtype=float) / width
    return np.convolve(y, kernel, mode='same')


def safe_ratio(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    return np.asarray(num, dtype=float) / np.clip(np.asarray(den, dtype=float), 1e-30, None)


def band_error(kbins: np.ndarray, ratio: np.ndarray) -> dict[str, float]:
    finite = np.where(np.isfinite(kbins) & np.isfinite(ratio) & (ratio > 0))[0]
    if len(finite) == 0:
        return {'low k': np.nan, 'mid k': np.nan, 'high k': np.nan}
    thirds = np.array_split(finite, 3)
    out = {}
    for name, idx in zip(['low k', 'mid k', 'high k'], thirds):
        out[name] = float(np.nanmean(np.abs(np.log10(np.clip(ratio[idx], 1e-30, None))))) if len(idx) else np.nan
    return out

if real is not None and snapshots:
    real_hist_arr = evenly_limit(real, MAX_REAL_HIST)
    real_pk_arr = evenly_limit(real, MAX_REAL_PK)

    real_hist_stats = field_histogram(real_hist_arr, bins=HIST_BINS)
    bin_edges = np.asarray(real_hist_stats['bin_edges'], dtype=float)
    hist_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    real_hist = np.asarray(real_hist_stats['hist'], dtype=float)

    pk_real, kbins = batch_power_spectra(real_pk_arr, nbins=PK_NBINS)
    pk_real_mean = np.nanmean(pk_real, axis=0)

    metrics_rows = []
    curves = []
    for row in snapshots:
        gen_nchw = to_nchw(row['samples'])
        centers, gen_hist = histogram_density(gen_nchw, bin_edges)
        width = float(np.mean(np.diff(bin_edges)))
        hist_l1 = float(np.nansum(np.abs(real_hist - gen_hist)) * width)

        pk_gen, _ = batch_power_spectra(gen_nchw, nbins=PK_NBINS)
        pk_gen_mean = np.nanmean(pk_gen, axis=0)
        ratio = safe_ratio(pk_gen_mean, pk_real_mean)
        log_mae = float(np.nanmean(np.abs(np.log10(np.clip(ratio, 1e-30, None)))))
        bands = band_error(kbins, ratio)

        metrics_rows.append({
            'stage': row['stage'],
            'epoch': row['epoch'],
            'n_generated': len(row['samples']),
            'onepoint_l1': hist_l1,
            'pk_log10_mae': log_mae,
            **bands,
        })
        curves.append({
            'stage': row['stage'],
            'epoch': row['epoch'],
            'hist': gen_hist,
            'pk_ratio': ratio,
            'pk_gen_mean': pk_gen_mean,
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_csv = OUT_DIR / 'learning_process_epoch_metrics.csv'
    metrics_df.to_csv(metrics_csv, index=False)
    display(metrics_df)
    print('wrote', metrics_csv)
else:
    metrics_df = pd.DataFrame()
    curves = []
    print('Skipped statistics: need both real reference and snapshot files.')


## Poster figure 1: statistics learned over training

This is the main plot I would use if the poster needs to explain the learning process. It shows that the same sampling procedure gives different statistics as the checkpoint improves.

In [ ]:
POSTER_RC = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 19,
    'axes.titlesize': 22,
    'axes.labelsize': 21,
    'xtick.labelsize': 17,
    'ytick.labelsize': 17,
    'legend.fontsize': 16,
    'figure.dpi': 140,
    'savefig.dpi': 300,
}

STAGE_STYLE = {
    'early':  {'color': '#D55E00', 'marker': 'o'},
    'middle': {'color': '#CC79A7', 'marker': 'D'},
    'late':   {'color': '#009E73', 'marker': '^'},
    'final':  {'color': '#0072B2', 'marker': 's'},
}
REAL_COLOR = '#20242A'
REF_COLOR = '#4A4A4A'


def selected_curves_for_overlay(curves: list[dict]) -> list[dict]:
    if len(curves) <= 3:
        return curves
    indices = [0, len(curves) // 2, len(curves) - 1]
    out = [curves[i].copy() for i in indices]
    out[0]['stage'] = 'early'
    out[1]['stage'] = 'middle'
    out[2]['stage'] = 'final'
    return out


def save_both(fig, name: str) -> tuple[Path, Path]:
    out = OUT_DIR / name
    poster_out = POSTER_DIR / name
    fig.savefig(out, bbox_inches='tight', dpi=300)
    fig.savefig(poster_out, bbox_inches='tight', dpi=300)
    print('wrote', out)
    print('wrote', poster_out)
    return out, poster_out


def plot_learning_stats() -> Path | None:
    if real is None or not curves:
        display(Markdown('Statistics plot skipped because real reference or snapshot files are missing.'))
        return None

    show_curves = selected_curves_for_overlay(curves)
    with plt.rc_context(POSTER_RC):
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), constrained_layout=True)
        ax_pdf, ax_pk = axes

        ax_pdf.plot(hist_centers, smooth(real_hist, 3), color=REAL_COLOR, lw=3.0, label='real reference')
        for c in show_curves:
            style = STAGE_STYLE.get(c['stage'], {'color': '#666666', 'marker': 'o'})
            label = f"{c['stage']} checkpoint"
            ax_pdf.plot(hist_centers, smooth(c['hist'], 3), color=style['color'], lw=2.8, label=label)
        ax_pdf.set_yscale('log')
        ax_pdf.set_xlabel('Normalized HI value')
        ax_pdf.set_ylabel('Pixel density')
        ax_pdf.set_title('One-point PDF', pad=10)
        ax_pdf.set_ylim(bottom=max(1e-5, np.nanmin(real_hist[real_hist > 0]) * 0.4))
        ax_pdf.grid(False)

        ax_pk.axhline(1.0, color=REF_COLOR, ls='--', lw=2.2, label='real reference')
        for c in show_curves:
            style = STAGE_STYLE.get(c['stage'], {'color': '#666666', 'marker': 'o'})
            ax_pk.plot(kbins, c['pk_ratio'], color=style['color'], lw=2.8, marker=style['marker'], ms=5.5, label=f"{c['stage']} checkpoint")
        ax_pk.set_xlabel('k bin')
        ax_pk.set_ylabel('Generated / real mean P(k)')
        ax_pk.set_title('Power-spectrum ratio', pad=10)
        finite_ratios = np.concatenate([np.asarray(c['pk_ratio'])[np.isfinite(c['pk_ratio'])] for c in show_curves])
        ymax = min(2.2, max(1.35, float(np.nanpercentile(finite_ratios, 95)) * 1.15)) if len(finite_ratios) else 1.6
        ax_pk.set_ylim(0.0, ymax)
        ax_pk.grid(False)

        for ax in axes:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_linewidth(1.25)
            ax.spines['bottom'].set_linewidth(1.25)
            ax.tick_params(width=1.15, length=5)

        handles, labels = ax_pdf.get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=len(handles), frameon=False)
        fig.suptitle('Training checkpoints with the same DPM-Solver-50 sampler', y=1.20, fontsize=25)
        _, poster_path = save_both(fig, 'learning_process_stats_poster.png')
        plt.show()
    return poster_path

learning_stats_plot = plot_learning_stats()


## Poster option: parallel P(k) panels

This version avoids the confusing ratio overlay. Each panel shows the same real mean P(k) and the generated mean P(k) from one checkpoint. The sampler is still fixed to DPM-Solver-50; the only change from left to right is training time.

In [ ]:
def plot_pk_parallel_panels() -> Path | None:
    if real is None or not curves:
        display(Markdown('Parallel P(k) plot skipped because real reference or snapshot files are missing.'))
        return None

    show_curves = selected_curves_for_overlay(curves)
    stage_title = {
        'early': 'Early checkpoint',
        'middle': 'Middle checkpoint',
        'late': 'Late checkpoint',
        'final': 'Final checkpoint',
    }

    with plt.rc_context(POSTER_RC):
        fig, axes = plt.subplots(
            1,
            len(show_curves),
            figsize=(4.65 * len(show_curves), 4.65),
            sharex=True,
            sharey=True,
            constrained_layout=True,
        )
        if len(show_curves) == 1:
            axes = [axes]

        positive_real = np.asarray(pk_real_mean)[np.asarray(pk_real_mean) > 0]
        ymin = max(1e-6, float(np.nanmin(positive_real)) * 0.45) if len(positive_real) else 1e-6
        ymax_candidates = [float(np.nanmax(pk_real_mean))]

        for ax, c in zip(axes, show_curves):
            style = STAGE_STYLE.get(c['stage'], {'color': '#666666', 'marker': 'o'})
            gen_pk = c.get('pk_gen_mean')
            if gen_pk is None:
                gen_pk = np.asarray(c['pk_ratio']) * np.asarray(pk_real_mean)
            gen_pk = np.asarray(gen_pk, dtype=float)
            ymax_candidates.append(float(np.nanmax(gen_pk)))

            ax.plot(kbins, pk_real_mean, color=REAL_COLOR, lw=3.2, label='real P(k)')
            ax.plot(
                kbins,
                gen_pk,
                color=style['color'],
                lw=3.2,
                marker=style['marker'],
                ms=5.2,
                markeredgecolor='white',
                markeredgewidth=0.8,
                label='generated P(k)',
            )
            ax.set_yscale('log')
            ax.set_title(f"{stage_title.get(c['stage'], c['stage'].title())}\nepoch {int(c['epoch'])}", pad=8)
            ax.set_xlabel('k bin')
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_linewidth(1.25)
            ax.spines['bottom'].set_linewidth(1.25)
            ax.tick_params(width=1.15, length=5)

        axes[0].set_ylabel('P(k)')
        ymax = max(ymax_candidates) * 1.8
        for ax in axes:
            ax.set_ylim(ymin, ymax)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False)
        fig.suptitle('Generated power spectra approach the real reference during training', y=1.34, fontsize=24)
        _, poster_path = save_both(fig, 'learning_process_pk_parallel_poster.png')
        plt.show()
    return poster_path

pk_parallel_plot = plot_pk_parallel_panels()


## Poster figure 2: low-k versus high-k learning

This plot is more explicit about Nick's point: if low-k structure is learned first, the large-scale error should drop earlier than the high-k error. The y-axis is an absolute log-ratio error, so **lower is better**.

In [ ]:
def plot_pk_band_errors() -> Path | None:
    if metrics_df.empty:
        display(Markdown('P(k)-band plot skipped because metrics are unavailable.'))
        return None

    band_cols = ['low k', 'mid k', 'high k']
    band_labels = {
        'low k': 'large scales\n(low k)',
        'mid k': 'intermediate\n(mid k)',
        'high k': 'small scales\n(high k)',
    }
    band_colors = {
        'low k': '#0072B2',
        'mid k': '#009E73',
        'high k': '#D55E00',
    }
    x = np.arange(len(metrics_df))
    labels = [f"{row.stage}\n(ep. {int(row.epoch)})" for row in metrics_df.itertuples()]

    with plt.rc_context(POSTER_RC):
        fig, ax = plt.subplots(figsize=(7.4, 5.2), constrained_layout=True)
        for col in band_cols:
            ax.plot(
                x,
                metrics_df[col].to_numpy(dtype=float),
                color=band_colors[col],
                lw=3.2,
                marker='o',
                ms=8.5,
                markeredgecolor='white',
                markeredgewidth=1.0,
                label=band_labels[col],
            )
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel(r'Mean $|\log_{10}(P_\mathrm{gen}/P_\mathrm{real})|$')
        ax.set_xlabel('Training checkpoint')
        ax.set_title('Which spatial scales are learned first?', pad=12)
        ax.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.25)
        ax.spines['bottom'].set_linewidth(1.25)
        ax.tick_params(width=1.15, length=5)
        ax.legend(loc='upper right', frameon=False, handlelength=2.2)
        _, poster_path = save_both(fig, 'learning_process_pk_band_errors.png')
        plt.show()
    return poster_path

pk_band_plot = plot_pk_band_errors()


## Optional image strip

This is intentionally secondary. It uses one generated sample from early, middle, and final checkpoints with one shared color scale. Use it only if the poster needs a visual example next to the statistics.

In [ ]:
def plot_optional_image_strip(sample_index: int = 0) -> Path | None:
    if not snapshots:
        return None
    chosen = selected_curves_for_overlay([{**row, 'hist': None, 'pk_ratio': None} for row in snapshots])
    # Recover sample arrays for chosen epochs.
    by_epoch = {row['epoch']: row for row in snapshots}
    chosen_rows = [by_epoch[c['epoch']] for c in chosen]
    all_imgs = np.concatenate([row['samples'] for row in chosen_rows], axis=0)
    vmin, vmax = np.percentile(all_imgs, [1.0, 99.0])

    with plt.rc_context(POSTER_RC):
        fig, axes = plt.subplots(1, len(chosen_rows), figsize=(7.8, 3.0), constrained_layout=True)
        if len(chosen_rows) == 1:
            axes = [axes]
        for ax, row, c in zip(axes, chosen_rows, chosen):
            idx = min(sample_index, len(row['samples']) - 1)
            ax.imshow(row['samples'][idx], cmap='viridis', vmin=vmin, vmax=vmax, interpolation='nearest')
            ax.set_title(f"{c['stage']}\nepoch {row['epoch']}", pad=8, fontsize=20)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
        _, poster_path = save_both(fig, 'learning_process_image_strip.png')
        plt.show()
    return poster_path

image_strip_plot = plot_optional_image_strip(sample_index=0)


## Saved poster files

The notebook writes every plot both to `results/nf_generalize_fig2/quickcheck/` and to `poster/figs/`.

In [ ]:
for path in [learning_stats_plot, pk_parallel_plot, pk_band_plot, image_strip_plot]:
    if path is not None and Path(path).exists():
        display(Markdown(f'### `{Path(path).name}`'))
        display(Image(filename=str(path)))
